### 3. Data Relabel

This notebook implements the **Sleep-EDF Relabeling Pipeline** for structured epoch-level labeling.

---

#### Processing Steps

- Load PSG EDF (signal data)
- Load matching Hypnogram EDF (sleep stage annotations)
- Select a single specified EEG channel (explicit selection)
- Align annotations to the PSG time base
- Segment recordings into 30-second epochs
- Assign one annotation per epoch
- Apply relabeling rules
- Save one Parquet file per PSG recording
- Generate a summary report for all processed recordings

---

#### Relabeling Rules

Each 30-second epoch contains:

- `raw_label` → Original Sleep-EDF annotation  
- `label` → Encoded class  
- `is_valid` → Evaluation mask  

Encoding:

- `?` (Unscored) → `label = 0`, `is_valid = False`
- Movement time → `label = 0`, `is_valid = False`
- Wake (W) → `label = 1`, `is_valid = True`
- N1, N2, N3/4, REM → `label = 2`, `is_valid = True`

Resulting class structure:

- `0` → Non-evaluable
- `1` → Wake
- `2` → Sleep (all sleep stages collapsed)

---

#### Epoch Retention Policy

- No epochs are removed.
- Movement and unscored epochs remain in the dataset.
- Non-evaluable epochs are excluded only during modeling via `is_valid`.

---

#### Output Per Recording

Each PSG recording produces one Parquet file containing:

- `epoch_index`
- `start`
- `end`
- `raw_label`
- `label`
- `is_valid`

---

#### Summary Report Output

After processing all recordings, a tab-separated summary report is written to: data/reports/<notebook_name>.txt

The report contains one row per PSG recording with the following fields:

- `#`
- `Subject`
- `PSG`
- `Hypnogram`
- `PSG_epochs`
- `All Epochs`
- `Valid Epochs`
- `Wake Epochs`
- `Sleep Epochs`
- `Invalid Epochs`

The report provides a structured audit of:

- Total epoch counts
- Valid vs. non-evaluable epochs
- Wake vs. sleep distribution per recording

---

#### Modeling Rule

Epochs where `is_valid = False` are excluded from:

- Model training
- Hyperparameter tuning
- Performance metric computation

These epochs remain stored to preserve temporal continuity but are treated as **Not Evaluated** during modeling.


In [ ]:
# --- IMPORTS ---

from __future__ import annotations

import os
import re
import numpy as np
import pandas as pd
import warnings
import mne

from pathlib import Path
from dataclasses import dataclass
from typing import Optional, Iterable, Tuple, List, Dict, Any, Union


In [ ]:
# --- CONFIGURATION CONSTANTS --- 

# Input folder that contains the raw (Sleep-EDF) data
INPUT_PATH = Path(r'../data/physionet-sleep-data')  # must contain *-PSG.edf and *-Hypnogram.edf
if not INPUT_PATH.exists():
    raise FileNotFoundError(f"Input path {INPUT_PATH} does not exist. Please create it and add the raw Sleep-EDF data before running this notebook.")

# Output folder for the processed epoch tables (Parquet files)
OUTPUT_DIR = Path(r'../data/epoch_tables')
if not OUTPUT_DIR.exists():
    raise FileNotFoundError(f"Output directory {OUTPUT_DIR} does not exist. Please create it before running this notebook.")    

# Process PSG files for signals, Hypnogram files for labels
PSG_GLOB = '**/*-PSG.edf'
HYPNOGRAM_GLOB = '**/*-Hypnogram.edf'

# Set the single channel name to use.
CHANNEL_NAME = 'EEG Fpz-Cz'

# Epoch duration in seconds. Common values are 20s, 30s, or 60s. This should match the epoch duration used in the dataset.
EPOCH_DURATION_SEC = 30.0

# Relabeling for binary classification (Wake vs Sleep)
# 0 = Wake, 1 = Sleep, and -1 = Invalid / Not evaluated (Movement, ?)
LABEL_MAPPING = {
    "Movement": -1,
    "?": -1,
    "W": 0,
    "N1": 1,
    "N2": 1,
    "N3": 1,
    "N4": 1,
    "REM": 1,
}

INVALID_LABEL_KEYS = ['Movement', '?'] # Will not be used in training, and ignored in summary statistics, but will be included in the output tables for completeness.

# Report directory for this notebook (matches downstream notebooks)
REPORT_DIR = (Path.cwd() / ".." / "data" / "reports").resolve()
if not REPORT_DIR.exists():
    raise FileNotFoundError(f"Report directory {REPORT_DIR} does not exist. Please create it before running this notebook.")

NOTEBOOK_NAME = "03_data_relabel" # This should match the name of this notebook (without the .ipynb extension)
LOG_FILE = REPORT_DIR / f"{NOTEBOOK_NAME}.txt"


In [ ]:
def build_epoch_boundaries(
    raw: mne.io.BaseRaw,
    epoch_duration: float = 30.0,
) -> Tuple[np.ndarray, np.ndarray]:
    """
    Return (epoch_starts, epoch_ends) in seconds, aligned to raw.times basis.
    """
    data_start = float(raw.times[0])
    data_end = float(raw.times[-1])

    # Number of epochs to cover the entire span [data_start, data_end]
    n_epochs = int(np.ceil((data_end - data_start) / epoch_duration))
    epoch_starts = np.arange(n_epochs, dtype=float) * epoch_duration + data_start
    epoch_ends = epoch_starts + float(epoch_duration)
    return epoch_starts, epoch_ends

In [ ]:
def annotations_to_intervals(
    ann: Union[mne.Annotations, List[Tuple[float, float, str]], Tuple[Tuple[float, float, str], ...]],
) -> List[Tuple[float, float, str]]:
    """
    Return list of (start, end, desc) in seconds.
    Accepts:
      - mne.Annotations (uses onset and duration)
      - list/tuple of (onset, duration, desc) OR (start, end, desc)
        NOTE: For list-like inputs, we treat the second value as DURATION if end < start,
              else treat as END. (This makes it tolerant to mixed conventions.)
    """
    intervals: List[Tuple[float, float, str]] = []

    if isinstance(ann, mne.Annotations):
        for onset, duration, desc in zip(ann.onset, ann.duration, ann.description):
            s = float(onset)
            e = float(onset + duration)
            intervals.append((s, e, str(desc)))
        return intervals

    if isinstance(ann, (list, tuple)):
        for item in ann:
            if not (isinstance(item, (list, tuple)) and len(item) == 3):
                raise ValueError("list-like ann must contain tuples (onset_or_start, duration_or_end, desc)")
            a0, a1, desc = item
            s = float(a0)
            v = float(a1)
            # Heuristic: if v <= s, assume it's a duration; otherwise assume it's an end time.
            e = float(s + v) if v <= s else float(v)
            intervals.append((s, e, str(desc)))
        return intervals
    
    raise ValueError("ann must be mne.Annotations or list-like of (onset_or_start, duration_or_end, desc)")

In [ ]:
def best_description_by_overlap(
    e_start: float,
    e_end: float,
    intervals: List[Tuple[float, float, str]],
) -> Optional[str]:
    """
    Return the annotation description with the largest overlap with [e_start, e_end).
    If there is no overlap, return None.
    """
    best_desc: Optional[str] = None
    best_ov: float = 0.0

    for a_start, a_end, desc in intervals:
        ov_start = max(e_start, a_start)
        ov_end = min(e_end, a_end)
        overlap = max(0.0, ov_end - ov_start)
        if overlap > best_ov:
            best_ov = overlap
            best_desc = desc

    return best_desc

In [ ]:
def normalize_sleep_edf_label(desc: Optional[str]) -> Optional[str]:
    """
    Deterministic mapping for Sleep-EDF hypnogram strings.
    No regex. Explicit and deterministic.

    Returns one of: 'W','N1','N2','N3','N4','REM','?','Movement', or None.
    """
    if desc is None:
        return None

    # normalize whitespace (no regex)
    d = " ".join(str(desc).strip().split())

    table = {
        "Sleep stage W": "W",
        "Sleep stage 1": "N1",
        "Sleep stage 2": "N2",
        "Sleep stage 3": "N3",
        "Sleep stage 4": "N4",
        "Sleep stage R": "REM",
        "Sleep stage ?": "?",
        "Movement time": "Movement",
    }

    if d not in table:
        raise ValueError(f"Unexpected hypnogram label: '{d}'")

    return table[d]

In [ ]:
def map_and_validate_label(
    raw_label: Optional[str],
    mapping: Dict[str, int],
) -> Tuple[Optional[int], bool]:
    """
    Convert normalized raw_label to integer class using mapping.
    Validity rule: label is valid iff mapped value != -1.
    """
    if raw_label is None:
        return None, False

    if raw_label not in mapping:
        provided = sorted(mapping.keys())
        raise ValueError(
            f"Unmapped label encountered: '{raw_label}'.\n"
            f"Provided mapping keys: {provided}"
        )

    y = mapping[raw_label]
    is_valid = (y != -1)
    return y, is_valid

In [ ]:
def annotations_to_epoch_labels(
    raw: mne.io.BaseRaw,
    ann: Union[mne.Annotations, List[Tuple[float, float, str]], Tuple[Tuple[float, float, str], ...]],
    epoch_duration: float = 30.0,
    mapping: Optional[Dict[str, int]] = None,
) -> pd.DataFrame:
    """
    Convert mne Annotations (or list-like intervals) to fixed-duration epochs and map labels.

    Returns a DataFrame with columns:
      - epoch_index
      - start
      - end
      - raw_label   (normalized string label)
      - label       (mapped integer label or None)
      - is_valid    (bool)
    """
    if mapping is None:
        raise ValueError("mapping must be provided. No default mapping is applied.")

    epoch_starts, epoch_ends = build_epoch_boundaries(raw, epoch_duration=epoch_duration)
    intervals = annotations_to_intervals(ann)

    rows: List[Dict[str, Any]] = []

    for idx, (s, e) in enumerate(zip(epoch_starts, epoch_ends)):
        desc = best_description_by_overlap(float(s), float(e), intervals)
        raw_label = normalize_sleep_edf_label(desc)
        label, is_valid = map_and_validate_label(raw_label, mapping)

        rows.append({
            "epoch_index": int(idx),
            "start": float(s),
            "end": float(e),
            "raw_label": raw_label,
            "label": label,
            "is_valid": bool(is_valid),
        })

    return pd.DataFrame(rows)

In [ ]:
def read_raw_edf_safely(edf_path: str, preload: bool = True):
    """
    Read EDF while suppressing common non-fatal MNE filter metadata warnings.
    """
    with warnings.catch_warnings():
        warnings.filterwarnings(
            "ignore",
            message="Channels contain different highpass filters.*",
            category=RuntimeWarning,
        )
        warnings.filterwarnings(
            "ignore",
            message="Channels contain different lowpass filters.*",
            category=RuntimeWarning,
        )
        warnings.filterwarnings(
            "ignore",
            message="Highpass cutoff frequency .* is greater than lowpass cutoff frequency .*",
            category=RuntimeWarning,
        )
        raw = mne.io.read_raw_edf(edf_path, preload=preload, verbose=False)
    return raw

In [ ]:
def set_annotations_clipped(raw: mne.io.BaseRaw, ann: mne.Annotations) -> None:
    """
    Safely attach annotations by clipping any that extend outside the raw data range.
    This prevents: 'Limited annotation(s) that were expanding outside the data range.'
    """
    if ann is None or len(ann) == 0:
        return

    t0 = float(raw.times[0])            # usually 0.0
    t1 = float(raw.times[-1])           # last time in seconds

    onsets = np.asarray(ann.onset, dtype=float)
    durations = np.asarray(ann.duration, dtype=float)
    desc = list(ann.description)

    ends = onsets + durations

    # Clip to [t0, t1]
    onsets_clipped = np.clip(onsets, t0, t1)
    ends_clipped = np.clip(ends, t0, t1)
    durations_clipped = np.maximum(0.0, ends_clipped - onsets_clipped)

    # Drop zero-length annotations after clipping
    keep = durations_clipped > 0.0
    if not np.any(keep):
        return

    ann2 = mne.Annotations(
        onset=onsets_clipped[keep],
        duration=durations_clipped[keep],
        description=[d for d, k in zip(desc, keep) if k],
        orig_time=ann.orig_time,
    )

    raw.set_annotations(ann2)

In [ ]:
def select_channel(raw, channel_name: str):
    """
    Select a single EEG channel.

    - If the channel exists -> return raw with only that channel.
    - If not -> print available channels and raise an error.
    """

    chs = list(raw.ch_names)

    if not chs:
        raise ValueError("No signal channels found (likely an annotations-only EDF).")

    if channel_name not in chs:
        print("\nRequested channel:", channel_name)
        print("Available channels:")
        for ch in chs:
            print("  -", ch)
        raise ValueError(f"Channel '{channel_name}' not found.")

    return raw.copy().pick([channel_name])

In [ ]:
def sanitize_for_parquet(df: pd.DataFrame) -> pd.DataFrame:
    """
    Make a DataFrame safer for Parquet serialization.

    Notes:
      - Avoid deprecated pandas dtype check helpers (pandas 3.x warnings).
      - Keep label columns stable and explicit.
    """
    out = df.copy()

    for col in out.columns:
        s = out[col]
        dtype = s.dtype

        # Period -> string (still a good guard, even if not present in your printed dtypes)
        if isinstance(dtype, pd.PeriodDtype):
            out[col] = s.astype(str)
            continue

        # tz-aware datetime -> UTC naive
        if isinstance(dtype, pd.DatetimeTZDtype):
            out[col] = s.dt.tz_convert("UTC").dt.tz_localize(None)
            continue

        # optional: normalize 'object' that is actually strings
        # (keeps it deterministic across runs)
        if dtype == object:
            # If it's clearly stringy, force to string (safe for Parquet)
            # Avoid converting mixed-type object columns unintentionally.
            if s.dropna().map(type).isin([str]).all() if len(s.dropna()) else False:
                out[col] = s.astype(str)

    return out

In [ ]:
def save_epoch_table(epoch_table: pd.DataFrame, out_path: str) -> None:
    """
    Save epoch table to Parquet robustly.

    Writes via pyarrow directly and strips pandas metadata to avoid
    extension-type registration conflicts like:
      'A type extension with name pandas.period already defined'
    """
    import sys
    import pyarrow as pa
    import pyarrow.parquet as pq

    df = sanitize_for_parquet(epoch_table)

    try:
        table = pa.Table.from_pandas(df, preserve_index=False)

        # Strip pandas metadata (key step)
        md = table.schema.metadata or {}
        md2 = {k: v for k, v in md.items() if k != b"pandas"}
        table = table.replace_schema_metadata(md2)

        pq.write_table(table, out_path)

    except Exception as e:
        print("\nPARQUET WRITE FAILED")
        print("Python exe:", sys.executable)
        print("pandas:", pd.__version__)
        print("pyarrow:", pa.__version__)
        print("Output path:", out_path)
        print("\nDataFrame dtypes:")
        print(df.dtypes)
        raise RuntimeError(f"Parquet write failed: {e}")


In [ ]:
def iter_files(input_path: Path, glob_pattern: str):
    input_path = Path(input_path)
    for p in sorted(input_path.glob(glob_pattern)):
        if p.is_file():
            yield p

In [ ]:
def record_key_from_stem(stem: str) -> str:
    """
    Pairing key for Sleep-EDF Expanded:
      PSG stem:       SC4012E0-PSG       -> token 'SC4012E0' -> key 'SC4012E'
      Hypnogram stem: SC4012EC-Hypnogram -> token 'SC4012EC' -> key 'SC4012E'
    """
    token = stem.split('-')[0]
    return token[:-1] if len(token) > 0 else token

In [ ]:
def extract_subject_from_path(psg_path: Path) -> str:
    """Best-effort subject extraction (Sleep-EDF is commonly organized as .../<SUBJECT>/<FILE>)."""
    parent = psg_path.parent.name
    if re.match(r"^SC\d{3}$", parent):
        return parent
    return parent  # fallback (keeps something meaningful)


In [ ]:
def write_summary_df_to_file(summary_df: pd.DataFrame, notebook_name, reports_root):
    """
    Write an existing summary_df to data/reports/<notebook_name>.txt (tab-separated + fixed-width).
    - Forces Subject = first 5 chars of PSG filename.
    - Enforces required column order.
    - Returns absolute path to written file.
    """

    df = summary_df.copy()

    if "PSG" not in df.columns:
        raise KeyError("Column 'PSG' not found in summary_df")

    # Auto-generate "#" column if missing
    if "#" not in df.columns:
        df["#"] = range(1, len(df) + 1)

    # Force Subject from PSG filename (e.g., SC4002E0-PSG.edf -> SC400)
    df["Subject"] = df["PSG"].astype(str).str[:5]

    ordered_cols = [
        "#",
        "Subject",
        "PSG",
        "Hypnogram",
        "PSG_epochs",
        "All Epochs",
        "Valid Epochs",
        "Wake Epochs",
        "Sleep Epochs",
        "Invalid Epochs",
    ]

    missing = [c for c in ordered_cols if c not in df.columns]
    if missing:
        raise KeyError(f"Missing required columns in summary_df: {missing}")

    df = df[ordered_cols]

    col_widths = {
        "#": 4,
        "Subject": 10,
        "PSG": 28,
        "Hypnogram": 28,
        "PSG_epochs": 34,
        "All Epochs": 12,
        "Valid Epochs": 14,
        "Wake Epochs": 13,
        "Sleep Epochs": 14,
        "Invalid Epochs": 15,
    }

    def format_row(row):
        return "\t".join(f"{str(row[c]):<{col_widths[c]}}" for c in ordered_cols)

    # Ensure output directory exists
    os.makedirs(reports_root, exist_ok=True)

    out_file = os.path.join(reports_root, f"{notebook_name}.txt")

    with open(out_file, "w") as f:
        header_line = "\t".join(f"{c:<{col_widths[c]}}" for c in ordered_cols)
        separator = "-" * len(header_line)

        f.write(header_line + "\n")
        f.write(separator + "\n")

        for _, row in df.iterrows():
            f.write(format_row(row) + "\n")

    return os.path.abspath(out_file)

In [ ]:
# --- FILE DISCOVERY AND VALIDATION ---

# Verify all expected files are present before processing. 
# This prevents silent failures later on and ensures that the dataset is complete and correctly structured.
try:
    psg_files = list(iter_files(INPUT_PATH, PSG_GLOB))
    hyp_files = list(iter_files(INPUT_PATH, HYPNOGRAM_GLOB))

    print(f"Found {len(psg_files)} PSG EDF(s).")
    print(f"Found {len(hyp_files)} Hypnogram EDF(s).")

    if len(psg_files) == 0:
        print("No PSG files found. Please check the input path and glob pattern.")
    elif len(hyp_files) != len(psg_files):
        raise ValueError("Warning: Number of Hypnogram files does not match number of PSG files. Check for missing or extra files.")
except Exception as e:
    raise RuntimeError(f"File discovery failed. Please check the input path and glob patterns.\nerror details: {e}")


In [ ]:
# --- BATCH PROCESSING ---

hyp_by_key = {record_key_from_stem(p.stem): p for p in hyp_files}


summary_rows = []

for psg_path in psg_files:
    try:
        key = record_key_from_stem(psg_path.stem)
        hyp_path = hyp_by_key.get(key)

        if hyp_path is None:
            raise FileNotFoundError(f"No hypnogram found for key={key} (PSG={psg_path.name})")

        subject = extract_subject_from_path(psg_path)

        # 1) Load PSG signals
        raw = read_raw_edf_safely(str(psg_path), preload=False)

        # 2) Select EEG channel (robust)
        raw_1ch = select_channel(raw, CHANNEL_NAME)

        # 3) Load hypnogram annotations (annotations-only EDF)
        ann = mne.read_annotations(str(hyp_path))

        # 4) Attach annotations to PSG time base
        set_annotations_clipped(raw_1ch, ann)

        # 5) Convert to 30s epoch labels
        epoch_table = annotations_to_epoch_labels(
            raw_1ch,
            raw_1ch.annotations,
            epoch_duration=EPOCH_DURATION_SEC,
            mapping=LABEL_MAPPING
        )

        # 6) Compute summary counts
        n_epochs = int(len(epoch_table))
        n_valid = int(epoch_table["is_valid"].sum())
        n_wake = int((epoch_table["label"] == 0).sum())
        n_sleep = int((epoch_table["label"] == 1).sum())
        n_invalid = int((epoch_table["label"] == -1).sum())

        # 7) Save
        out_path = OUTPUT_DIR / f"{psg_path.stem}_epochs.parquet"
        save_epoch_table(epoch_table, str(out_path))

        summary_rows.append({
            "Subject": subject,
            "PSG": psg_path.name,
            "Hypnogram": hyp_path.name,
            "PSG_epochs": out_path.name,
            "All Epochs": n_epochs,
            "Valid Epochs": n_valid,
            "Wake Epochs": n_wake,
            "Sleep Epochs": n_sleep,
            "Invalid Epochs": n_invalid,
        })

        # Simple progress indicator (one '#' per file)
        print("#", end="")

    except Exception as e:
        print(f"FAILED: {psg_path}")
        print("  Error:", str(e))
        summary_rows.append({"PSG": str(psg_path), "error": str(e)})
        break


In [ ]:
# --- SUMMARY REPORT ---
summary_df = pd.DataFrame(summary_rows)
path = write_summary_df_to_file(summary_df, NOTEBOOK_NAME, REPORT_DIR)
print(f"Summary saved to: {path}")